# REGPLEX_Local — REGPLEX v12
Publication-oriented local notebook for di-centric perplexity valley analysis.


## 1. Scientific Introduction
REGPLEX v12 uses dinucleotide perplexity as the primary detection signal and retains mono/tri as supporting evidence layers.


## 2. Algorithm Overview
The pipeline computes mono/di/tri perplexities once, runs a di-centric multi-scale LPC detection engine by default, applies Kadane refinement, persistence/prominence filtering, NMS, and merging, then reports support annotations from all layers.


## 3. Loading FASTA
Load FASTA and inspect sequence metadata before analysis.


## 4. Mono Perplexity
Compute mononucleotide perplexity profile.


## 5. Di Perplexity
Compute dinucleotide perplexity profile (primary layer).


## 6. Tri Perplexity
Compute trinucleotide perplexity profile.


## 7. Why Dinucleotide Perplexity Is the Primary Signal
Dinucleotide perplexity captures nearest-neighbour stacking interactions and local duplex stability, which are central to promoter architecture and DNA structural organization.
Mono and Tri layers are retained as support for interpretability and cross-species comparison, but they are not used as primary prediction signals in default promoter/genome modes.


## 8. Savitzky–Golay Smoothing
Apply a single Savitzky–Golay pass to each perplexity profile to denoise while preserving valley geometry.


## 9. Multi-scale Landscapes
Generate landscapes at 25, 50, 100, 200, and 400 bp from the smoothed profiles.


## 10. Consensus LPC
Compute scale-wise LPC, normalize each scale independently, then median-combine to layer consensus and final ConsensusLPC.


## 11. Candidate Generation
Generate all candidate valleys from contiguous positive ConsensusLPC runs and retain their raw metrics for later ranking.


## 12. Kadane Refinement
Refine each candidate valley by locating the strongest continuous core between 50 and 1000 bp.


## 13. Persistence, Prominence, and NMS
Apply hard persistence filtering (≥ 0.80), adaptive prominence filtering, and non-maximum suppression to remove weak and redundant calls.


## 14. Valley Merging and Ranking
Merge nearby surviving valleys when the gap is below 25 bp, then compute raw and normalized valley scores.


## 15. Visualization
Generate publication-ready figures showing raw/smoothed perplexity, ConsensusLPC, candidate valleys, Kadane cores, and final merged valleys.


## 16. Motif Annotation
Annotate detected valleys with regex/IUPAC motifs after final filtering.


## 17. Export Results
Export ranked valley outputs for downstream biological interpretation.


## 18. Final Biological Interpretation
Interpret the top-ranked merged valleys as the highest-confidence regulatory domains and review motif content, prominence, persistence, and support before downstream validation.


In [ ]:
import pandas as pd
from regplex_core import (
    parse_fasta,
    analyze_sequence,
    domains_dataframe,
    compute_mono_perplexity,
    compute_di_perplexity,
    compute_tri_perplexity,
)
from motif_engine import compile_motifs, annotate_domains
from visualization import (
    plot_perplexity_layers,
    plot_smoothed_perplexity,
    plot_layer_consensus,
    plot_consensus_lpc,
    plot_kadane_domains,
    plot_domain_ranking,
    plot_motif_architecture,
)


In [ ]:
with open('examples/ecoli.fasta', encoding='utf-8') as fh:
    records = parse_fasta(fh.read())
header, seq = records[0]
print(header, len(seq))


In [ ]:
mono = compute_mono_perplexity(seq)
di = compute_di_perplexity(seq)
tri = compute_tri_perplexity(seq)
len(mono), len(di), len(tri)


In [ ]:
result = analyze_sequence(
    header,
    seq,
    mode='promoter',
    scales=[25, 50, 100, 200, 400],
    landscape_method='mean',
    normalization_method='robust_z',
    ensemble_method='median',
    core_window_upstream=500,
    core_window_downstream=200,
    reference_point=0,
    sg_window_length=21,
    sg_poly_order=3,
)
df = domains_dataframe([result])
df.head()


In [ ]:
plot_perplexity_layers(result.mono, result.di, result.tri).show()
plot_smoothed_perplexity(
    result.mono, result.di, result.tri,
    result.smoothed_mono, result.smoothed_di, result.smoothed_tri,
    result.domains,
).show()
plot_layer_consensus(result.layer_consensus, result.domains).show()
plot_consensus_lpc(result.consensus_lpc, result.domains).show()
plot_kadane_domains(result.consensus_lpc, result.domains, result.kadane_core, result.candidates).show()
plot_domain_ranking(result.domains).show()


In [ ]:
motifs = compile_motifs('TATAWAWR\nGGGN{1,7}GGG')
annotate_domains(result.domains, motifs)
plot_motif_architecture(result.domains).show()
domains_dataframe([result]).head()


In [ ]:
out = domains_dataframe([result])
out.to_csv('examples/sample_output.csv', index=False)
out[['ID','Start','End','ValleyScore','OverallSupport']].head()
